In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from embedder import load_phobert, embed_texts
from embedding_cache import get_or_create_embeddings
from cso_optimizer import optimize_cso
from trainer import train_svm, evaluate
from calibration import fit_calibrated_svm, predict_with_confidence, save_calibrator

In [ ]:
BASE_DIR = "/kaggle/working/models"
PHOBERT_DIR = f"{BASE_DIR}/phobert"
EMBEDDING_DIR = f"{BASE_DIR}/cache_embeddings"
MODEL_PATH = f"{BASE_DIR}/final_model.joblib"
BEST_C_PATH = f"{BASE_DIR}/best_c.json"
FITNESS_CACHE_PATH = f"{BASE_DIR}/fitness_cache.json"
CALIBRATOR_PATH = f"{BASE_DIR}/calibrated_svm.joblib"

In [ ]:
train_df = pd.read_csv("/kaggle/input/your-data/train.csv")
dev_df = pd.read_csv("/kaggle/input/your-data/dev.csv")
test_df = pd.read_csv("/kaggle/input/your-data/test.csv")

train_texts = train_df["title"].tolist()
train_labels = train_df["label"].tolist()

dev_texts = dev_df["title"].tolist()
dev_labels = dev_df["label"].tolist()

test_texts = test_df["title"].tolist()
test_labels = test_df["label"].tolist()

In [ ]:
tokenizer, phobert_model = load_phobert(PHOBERT_DIR)

X_train, y_train = get_or_create_embeddings(
    train_texts, train_labels,
    embed_texts, tokenizer, phobert_model,
    EMBEDDING_DIR, "train"
)

X_dev, y_dev = get_or_create_embeddings(
    dev_texts, dev_labels,
    embed_texts, tokenizer, phobert_model,
    EMBEDDING_DIR, "dev"
)

X_test, y_test = get_or_create_embeddings(
    test_texts, test_labels,
    embed_texts, tokenizer, phobert_model,
    EMBEDDING_DIR, "test"
)  

In [ ]:
if os.path.exists(BEST_C_PATH):
    with open(BEST_C_PATH, "r", encoding="utf-8") as f:
        best_C = json.load(f)["best_C"]
    print("Loaded best_C:", best_C)
else:
    best_C = optimize_cso(
        X_train=X_train,
        y_train=y_train,
        X_dev=X_dev,
        y_dev=y_dev,
        cache_path=FITNESS_CACHE_PATH,
        P=8,
        Tmax=5,
        sample_size=100000,
        random_state=42
    )

    os.makedirs(os.path.dirname(BEST_C_PATH), exist_ok=True)
    with open(BEST_C_PATH, "w", encoding="utf-8") as f:
        json.dump({"best_C": float(best_C)}, f, ensure_ascii=False, indent=2)

    print("Saved best_C:", best_C)

In [ ]:
svm_model = train_svm(X_train, y_train, C=best_C, random_state=42)
dev_result = evaluate(svm_model, X_dev, y_dev, name="Dev")
test_result = evaluate(svm_model, X_test, y_test, name="Test")
joblib.dump(svm_model, MODEL_PATH)

In [ ]:
calibrator = fit_calibrated_svm(svm_model, X_dev, y_dev, method="sigmoid")
save_calibrator(calibrator, CALIBRATOR_PATH)

y_pred_cal, y_proba_cal, confidence_cal = predict_with_confidence(calibrator, X_test)

print("Sample confidence:", confidence_cal[:10])